# 9B. Pose Transformer Augmentation Ablation Colab


## Question, Scope, Interpretation

This stage isolates one question:

- do the current Stage 6-style pose augmentations improve validation behavior for the pose transformer?

The architecture, exercise subset, and training schedule stay fixed. Only augmentation changes.

Controlled conditions:
- same generic Stage 5 pose-sequence input
- same transformer config
- same selected exercises: `squat`, `pull_up`, `push_up`
- same balanced-count sampler
- same loss and early-stopping settings

Interpretation:
- if augmentation lowers `MAE` or improves `Within-1`, it is helping generalization
- if augmentation hurts consistently, it should be reduced or disabled for the transformer branch
- for `squat`, keep in mind that both variants are still generic-pose runs, not the dedicated squat branch


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TCN_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
TRANSFORMER_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_transformer.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')


def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [TCN_TRAIN_REL, TRANSFORMER_TRAIN_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'
TRAINING_OUTPUTS = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs'

print('POSE_INDEX =', POSE_INDEX)
print('Base pose trainer exists =', (DRIVE_PROJECT_ROOT / TCN_TRAIN_REL).exists())
print('Transformer trainer exists =', (DRIVE_PROJECT_ROOT / TRANSFORMER_TRAIN_REL).exists())


## Preset And Augmentation Variants


In [ ]:
import pandas as pd

RUNS = [
    {
        'exercise': 'squat',
        'seq_len': 256,
        'pose_run': 'pose_count_tcn_squat_seq256',
        'pose_best_run': 'squat_tcn_l1_channels96',
        'variants': {
            'aug_off': 'pose_transformer_squat_seq256_aug_off',
            'aug_on': 'pose_transformer_squat_seq256_aug_on',
        },
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'pose_run': 'pose_count_tcn_pull_up_seq192',
        'variants': {
            'aug_off': 'pose_transformer_pull_up_seq192_aug_off',
            'aug_on': 'pose_transformer_pull_up_seq192_aug_on',
        },
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'pose_run': 'pose_count_tcn_push_up_seq128',
        'variants': {
            'aug_off': 'pose_transformer_push_up_seq128_aug_off',
            'aug_on': 'pose_transformer_push_up_seq128_aug_on',
        },
    },
]

AUGMENTATION_VARIANTS = {
    'aug_off': {
        'time_warp_range': '0.0',
        'feature_noise_std': '0.0',
        'frame_dropout_prob': '0.0',
        'camera_motion_std': '0.0',
        'camera_zoom_std': '0.0',
        'joint_occlusion_prob': '0.0',
        'joint_occlusion_min_ratio': '0.10',
        'joint_occlusion_max_ratio': '0.30',
        'joint_occlusion_max_joints': '2',
    },
    'aug_on': {
        'time_warp_range': '0.12',
        'feature_noise_std': '0.02',
        'frame_dropout_prob': '0.03',
        'camera_motion_std': '0.0',
        'camera_zoom_std': '0.0',
        'joint_occlusion_prob': '0.0',
        'joint_occlusion_min_ratio': '0.10',
        'joint_occlusion_max_ratio': '0.30',
        'joint_occlusion_max_joints': '2',
    },
}

MODEL_DIM = 192
NUM_HEADS = 6
NUM_LAYERS = 4
FF_DIM = 384
DROPOUT = 0.20

meta_df = pd.read_csv(POSE_INDEX)
subset_df = meta_df[meta_df['type'].isin([cfg['exercise'] for cfg in RUNS])]
display(subset_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index())
print('AUGMENTATION_VARIANTS =', AUGMENTATION_VARIANTS)
print('Transformer =', {
    'model_dim': MODEL_DIM,
    'num_heads': NUM_HEADS,
    'num_layers': NUM_LAYERS,
    'ff_dim': FF_DIM,
    'dropout': DROPOUT,
})


## Train Augmentation-Off And Augmentation-On Runs


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in RUNS:
    for aug_name, run_name in cfg['variants'].items():
        aug_cfg = AUGMENTATION_VARIANTS[aug_name]
        cmd = [
            'python', '-u', str(DRIVE_PROJECT_ROOT / TRANSFORMER_TRAIN_REL),
            '--project-dir', str(DRIVE_PROJECT_ROOT),
            '--index-csv', str(POSE_INDEX),
            '--run-name', run_name,
            '--exercise', cfg['exercise'],
            '--seq-len', str(cfg['seq_len']),
            '--epochs', '80',
            '--batch-size', '16',
            '--lr', '0.0005',
            '--weight-decay', '0.0001',
            '--model-dim', str(MODEL_DIM),
            '--num-heads', str(NUM_HEADS),
            '--num-layers', str(NUM_LAYERS),
            '--ff-dim', str(FF_DIM),
            '--dropout', str(DROPOUT),
            '--patience', '15',
            '--loss', 'l1',
            '--eval-transform', 'raw',
            '--selection-metric', 'mae',
            '--sampler', 'balanced_count',
            '--time-warp-range', aug_cfg['time_warp_range'],
            '--feature-noise-std', aug_cfg['feature_noise_std'],
            '--frame-dropout-prob', aug_cfg['frame_dropout_prob'],
            '--camera-motion-std', aug_cfg['camera_motion_std'],
            '--camera-zoom-std', aug_cfg['camera_zoom_std'],
            '--joint-occlusion-prob', aug_cfg['joint_occlusion_prob'],
            '--joint-occlusion-min-ratio', aug_cfg['joint_occlusion_min_ratio'],
            '--joint-occlusion-max-ratio', aug_cfg['joint_occlusion_max_ratio'],
            '--joint-occlusion-max-joints', aug_cfg['joint_occlusion_max_joints'],
            '--device', 'cuda',
        ]
        print('\nRunning:', ' '.join(cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            training_failures.append({
                'exercise': cfg['exercise'],
                'augmentation': aug_name,
                'run_name': run_name,
                'returncode': exc.returncode,
            })
            print(f"FAILED: {cfg['exercise']} / {aug_name} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All augmentation ablation runs completed.')


## Validation Metric Review


In [ ]:
import json
import pandas as pd

rows = []
for cfg in RUNS:
    for aug_name, run_name in cfg['variants'].items():
        metrics_path = TRAINING_OUTPUTS / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'augmentation': aug_name,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(['exercise', 'augmentation']))

pivot_df = compare_df.pivot(index=['exercise', 'seq_len'], columns='augmentation', values=['valid_mae', 'valid_within_1', 'valid_rmse'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'valid_mae_aug_on' in pivot_df.columns and 'valid_mae_aug_off' in pivot_df.columns:
    pivot_df['delta_mae_aug_on_minus_aug_off'] = pivot_df['valid_mae_aug_on'] - pivot_df['valid_mae_aug_off']
if 'valid_rmse_aug_on' in pivot_df.columns and 'valid_rmse_aug_off' in pivot_df.columns:
    pivot_df['delta_rmse_aug_on_minus_aug_off'] = pivot_df['valid_rmse_aug_on'] - pivot_df['valid_rmse_aug_off']
if 'valid_within_1_aug_on' in pivot_df.columns and 'valid_within_1_aug_off' in pivot_df.columns:
    pivot_df['delta_within_1_aug_on_minus_aug_off'] = pivot_df['valid_within_1_aug_on'] - pivot_df['valid_within_1_aug_off']
display(pivot_df.sort_values('exercise'))

reference_rows = []
for cfg in RUNS:
    reference_variants = [('pose_tcn', cfg['pose_run'])]
    if cfg.get('pose_best_run'):
        reference_variants.append(('pose_best_squat', cfg['pose_best_run']))
    for variant, run_name in reference_variants:
        metrics_path = TRAINING_OUTPUTS / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        reference_rows.append({
            'exercise': cfg['exercise'],
            'reference_variant': variant,
            'run_name': run_name,
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

if reference_rows:
    display(pd.DataFrame(reference_rows).sort_values(['exercise', 'reference_variant']))


## Compare Both Variants Against The Trivial Baseline


In [ ]:
import json
import subprocess
import pandas as pd

comparison_failures = []
comparison_rows = []
for cfg in RUNS:
    for aug_name, run_name in cfg['variants'].items():
        run_dir = TRAINING_OUTPUTS / run_name
        predictions_csv = run_dir / 'predictions.csv'
        summary_json = run_dir / 'baseline_comparison_summary.json'
        rows_csv = run_dir / 'baseline_comparison_rows.csv'
        cmd = [
            'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
            '--index-csv', str(POSE_INDEX),
            '--predictions-csv', str(predictions_csv),
            '--exercise', cfg['exercise'],
            '--output-json', str(summary_json),
            '--output-csv', str(rows_csv),
        ]
        print('\nRunning:', ' '.join(cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            comparison_failures.append({
                'exercise': cfg['exercise'],
                'augmentation': aug_name,
                'run_name': run_name,
                'returncode': exc.returncode,
            })
            continue

        with open(summary_json, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        comparison_rows.append({
            'exercise': cfg['exercise'],
            'augmentation': aug_name,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        })

if comparison_failures:
    display(pd.DataFrame(comparison_failures))

display(pd.DataFrame(comparison_rows).sort_values(['exercise', 'augmentation']))
